# DVS Test Procedures - Holography

* Ku-band: alignment, errorbeam
* Band2: alignment, beam patterns & polarisation
* S-band: alignment, beam patterns & polarisation

                                                              As on 17/06/2026

In [ ]:
%matplotlib inline
import pylab as plt
import numpy as np

In [ ]:
from dvs import util, hologreport
from analysis import katselib

In [ ]:
ant = "e116"

In [ ]:
# A "sequential(2)" i.e. monotonic in lightness but with a kink around mid-range for emphasis.
# This seems more appropriate for showing 'devmaps' than the default offered by katholog.
CM = plt.cm.afmhot
CM = plt.cm.viridis

F_mr, f_eq = 5.8535, 8.507 # Derived from SKA-TEL-DSH-0000018 rev 2 & DS Design Report 316-000000-022 - EM Sensitivity Analysis

# PREDICTED patterns generated rel. to secondary focus (F0 in SKA-TEL-DSH-0000018 rev 2), shift to center on Q0
pDISHPARAMS = dict(telescope="SKA", xyzoffsets=[0.0, 1.476396+8.04, 0], xmag=-f_eq/F_mr, focallength=F_mr)

# Parameters for MEASURED patterns
# katholog x=cross-elevation=no offset,
#          y=elevation direction=el_axis_to_centre_on_main_reflector,
#          z=beam direction = az_axis_to_centre_on_main_reflector (from 316-000000-022 rev 1)
DISHPARAMS = dict(telescope="SKA", xyzoffsets=[0.0, 1.5363, 0.2330], xmag=-f_eq/F_mr, focallength=F_mr)

predL = hologreport.ResultSet("predicted", f_MHz=[], beacon_pol=[], clipextent=24.0) # Max extent 120deg!
predS = None
predK = hologreport.ResultSet("predicted", f_MHz=[], beacon_pol=[], clipextent=2.8) # Max extent 2.8deg
predKc = hologreport.ResultSet("predicted", f_MHz=[], beacon_pol=[], clipextent=1)

# Ku-Band

## Predicted Data


In [5]:
!ls ../models/beam-patterns/ska/Ku*

../models/beam-patterns/ska/Ku_45_11452.mat
../models/beam-patterns/ska/Ku_45_11697.mat
../models/beam-patterns/ska/Ku_45_11699.mat
../models/beam-patterns/ska/Ku_45_11700.mat
../models/beam-patterns/ska/Ku_45_12179.mat
../models/beam-patterns/ska/Ku_45_12251.mat
../models/beam-patterns/ska/Ku_45_12501_5.mat


In [ ]:
for f_MHz in [11452,11699, # X1
              #12179,12501.5 # X2
             ]:
    b, aH, aV = hologreport.load_predicted(f_MHz, None, pDISHPARAMS, el_deg=45, clipextent=predKc.clipextent, gridsize=512)
    predKc.f_MHz.append(f_MHz); predKc.beacon_pol.append(None)
    predKc.beams.append(b); predKc.apmapsH.append(aH); predKc.apmapsV.append(aV)

In [ ]:
# Tilt angles calculated from geo orbital slots, extra tilt and relative to MeerKAT center
geotilt = {"EUTELSAT 36D": (36.1,+3.535), # Extra +3.535 tilt for most Eutelsat
           "BADR-7": (26.0,0),
           "EUTELSAT 21B": (21.6,+3.535),
           "EUTELSAT 10B": (10.0,+3.535),
           "EUTELSAT 9B": (9.0,+3.535),
           "INTELSAT 37E (IS-37E)": (-18,0),
           "INTELSAT 25 (IS-25)": (-31.5,0),
         }
linpol = lambda pol, satellite_ID: hologreport.e_bn(pol, geotilt[satellite_ID])

In [ ]:
# Predicted maps for Ku-band
for f_MHz,pol in [(11452,"LCP"),(11452,"RCP"),(11700,"RCP"),(11700,"LCP"), # X1 typical
                  (12501.5,"RCP"),(12501.5,"LCP"), # X2 typical
                  #(12251,"RCP"),
                  #(12251,linpol("V","INTELSAT 25 (IS-25)")),
                 ]:
    b, aH, aV = hologreport.load_predicted(f_MHz, pol, pDISHPARAMS, el_deg=45, clipextent=predK.clipextent, gridsize=512)
    predK.f_MHz.append(f_MHz); predK.beacon_pol.append(pol)
    predK.beams.append(b); predK.apmapsH.append(aH); predK.apmapsV.append(aV)

## Measured Data

In [ ]:
### Ku-band
# Before final FI adjustment
rr = katselib.ls_archive(f"InstructionSet:\"scan-ant*{ant}*\"~1 AND StartTime:[2025-01-20T0:0:0Z TO 2025-10-01T0:0:0Z] AND CenterFrequency:[11000000000 TO *] AND InstructionSet:*holography*",
                    min_duration=1800, fields=["CaptureBlockId","StartTime","CenterFrequency",'Description','InstructionSet'], field_len=250)
print()
# Final FI adjustment and equivalents
rr += katselib.ls_archive(f"InstructionSet:\"scan-ant*{ant}*\"~1 AND StartTime:[2025-10-01T0:0:0Z TO *] AND CenterFrequency:[11000000000 TO *] AND InstructionSet:*holography*",
                    min_duration=1800, fields=["CaptureBlockId","StartTime","CenterFrequency",'Description','InstructionSet'], field_len=250)

In [ ]:
# Careful with the following!
# - HISPASAT 36W-1 large EB and H-V maps show clear pol issue, possibly because beacon is LCP/H. Omit until resolved!
# - EUTELSAT 36D gives consistently "different" results on all dishes - possibly its highly shaped beam?

In [ ]:
# With FI @ 103.37deg
ms_Ku0 = [
    hologreport.ResultSet(1760732651,f_MHz=[12501],beacon_pol=["LCP"],clipextent=3,tags=["FI0","night","EUTELSAT 8 West B"]),
    hologreport.ResultSet(1762911619,f_MHz=[12501],beacon_pol=["RCP"],clipextent=3,tags=["FI0","night","INTELSAT 37E"]),
    hologreport.ResultSet(1763182340,f_MHz=[11452],beacon_pol=["RCP"],clipextent=3,tags=["FI0","day","sunrise","INTELSAT 17"]),
]
ms_Ku0_hires = [
    hologreport.ResultSet(1764041269,f_MHz=[11452],beacon_pol=["RCP"],clipextent=6,tags=["FI0","night","extralmoffset=auto","INTELSAT 28"]),
]
msc_Ku0 = [ # X1, matching predicted patterns
    hologreport.ResultSet(1762917802,f_MHz=[11452,11700],beacon_pol=None,clipextent=1,tags=["FI0","night","3C 2739"]),
    hologreport.ResultSet(1763052886,f_MHz=[11452,11700],beacon_pol=None,clipextent=1,tags=["FI0","day","sunset","3C 454.3"]),
]


# FI @ 103.87deg
ms_Ku1 = [
    hologreport.ResultSet(1774724633,f_MHz=[11451],beacon_pol=["RCP"],clipextent=3,tags=["FI1","night","SES-4"]),
    hologreport.ResultSet(1774732225,f_MHz=[11452],beacon_pol=["LCP"],clipextent=3,tags=["FI1","night","HISPASAT 36W-1"]),
    hologreport.ResultSet(1774737774,f_MHz=[11449,11700],beacon_pol=["LCP","RCP"],clipextent=3,tags=["FI1","night","AMOS-3"]),
]
ms_Ku1_hires = [
    hologreport.ResultSet(1775939749,f_MHz=[11449,11700],beacon_pol=["LCP","RCP"],clipextent=10,tags=["FI1","0.14dps","night","AMOS-3"]),
    hologreport.ResultSet(1775950769,f_MHz=[11451],beacon_pol=["RCP"],clipextent=10,tags=["FI1","0.14dps","night","SES-4"]),
]
msc_Ku1 = [ # X1, matching predicted patterns
    hologreport.ResultSet(1776116944,f_MHz=[11452,11700],beacon_pol=None,clipextent=1,tags=["FI1","night","3C 279"]),
]

In [ ]:
# Check for channels, bandwidth & polswap.
# CAUTION: this often gets the signs of RCP/LCP wrong - possibly due to delay model?
for ms in ms_Ku0_hires:
    ds = util.open_dataset(ms.fid, ref_ant=ant, cache_root="./l1_data")
    katselib.plot_signalpol(ds, f_MHz=ms.f_MHz, pol=ms.beacon_pol, compscans='holo,track',scans='track', ant=ds.obs_params['track_ants'].split(',')[0])
    katselib.plot_signalpol(ds, f_MHz=ms.f_MHz, pol=ms.beacon_pol, compscans='holo,track',scans='track', ant=ds.obs_params['scan_ants'].split(',')[0])
    plt.title(ms.tags[-1], loc="left")

In [ ]:
# Find timingoffsets to use for these datasets. This may require some iteration!
for ms in ms_Ku1:
    hologreport.check_timingoffset(cached_url(ms.fid), ms.f_MHz, ant_, timingoffset=[0,0.02,0.04,0.08], extent=None)

In [ ]:
# Double-check the final value before loading the data
# - choose the smallest delta that increases amplitude SNR significantly AND doesn't increasing phase RMS significantly
for ms in ms_Ku1:
    if (ms.timingoffset != 0):
        hologreport.check_timingoffset(cached_url(ms.fid), ms.f_MHz, ant, timingoffset=[ms.timingoffset-0.02,ms.timingoffset,ms.timingoffset+0.02], extent=0.5)

In [ ]:
### Load the data

# Beacon signals
dMHz = 0.1 # ~1 channel i.e. highest SNR on beacon
# datasets = ms_Ku0_hires+ms_Ku1_hires; clipextent = np.inf
# datasets = ms_Ku1; clipextent = predK.clipextent
datasets = ms_Ku0_cycles; clipextent = predK.clipextent

# Continuum
# datasets, dMHz = msc_Ku0+msc_Ku1, 110; clipextent = predKc.clipextent # ~ 1% fractional BW in X1
# datasets, dMHz = msc_Sband1, 30 # ~ 1% fractional BW in S4, at most 1.7% in S0
# datasets, dMHz = msc_Lband1, 10; clipextent = predLc.clipextent # ~ 1% fractional BW

for ms in datasets:
    # ms.beams.clear(); ms.apmapsH.clear(); ms.apmapsV.clear() # Un-comment to reload all
    if (len(ms.beams) > 0): continue
    
    if True: # Not caching
        ds = util.open_dataset(ms.fid); cached_url = util.cbid2url
    else: # Caching
        ds = util.open_dataset(ms.fid, cache_root="./l1_data"); cached_url = lambda cbid: f"./l1_data/{cbid}/{cbid}_sdp_l0.full.rdb"

    b, aH, aV = hologreport.load_data(cached_url(ms.fid), ms.f_MHz, ant, DISHPARAMS, polswap=ms.polswap, timingoffset=ms.timingoffset,
                                      clipextent=min(ms.clipextent,clipextent), loadscan_cycles=ms.cycles, overlap_cycles=ms.overlap_cycles,
                                      findwrap="findwrap" in ms.tags, extralmoffset='auto' if "extralmoffset=auto" in ms.tags else [0,0],
                                      dMHz=dMHz, gridsize=512, flag_slew=True, flags_hrs=ms.flags_hrs, ignoreantennas=ms.ignoreantennas)
    ms.beams.extend(b); ms.apmapsH.extend(aH); ms.apmapsV.extend(aV)
    if ms.fid>1770000000: # TODO: TEMPORARY HACK
        ds.del_cache()

## Reports - High Res

In [ ]:
ku_hr = hologreport.generate_results(ms_Ku0_hires, predK,DF=10, eb_extent=(-0.3,0.3), cmap=CM, makepdfs=False,
                                     SNR_min=20, phaseRMS_max=40); # Ideally use 30,30

In [ ]:
ku_hr1 = hologreport.generate_results(ms_Ku1_hires, predK,DF=10, eb_extent=(-0.3,0.3), cmap=CM, makepdfs=False,
                                     SNR_min=20, phaseRMS_max=40); # Ideally use 30,30

In [ ]:
_ku = hologreport.filter_results(ku_hr, enviro_filter=lambda enviro: max(enviro['wind_mps']) < 5)

hologreport.meta_report(_ku, ["FI0"])

## Report - Cycles

In [ ]:
ku_cycl = hologreport.generate_results([_ for _ in ms_Ku0_cycles if (_.cycles is not None)], predK,DF=10, eb_extent=(-0.3,0.3), cmap=CM, makepdfs=False,
                                       SNR_min=10, phaseRMS_max=60);

In [ ]:
_ku = hologreport.filter_results(ku_cycl, enviro_filter=lambda enviro: max(enviro['wind_mps']) <= 5)

In [ ]:
hologreport.plot_vs_hod([_ku["FI0"]], ["FI0"], separate_freqs=False)

In [ ]:
hologreport.plot_errbeam_el([_ku["FI0"]], ["FI0 cycles"])

In [ ]:
hologreport.plot_errbeam_cycles([ms for ms in ms_Ku4 if (ms.fid == 1738442363)], predK, extent=(-0.2,0.2), clim=(-0.03,0.03))

## Reports - Low Res

In [ ]:
ku0 = hologreport.generate_results(ms_Ku0, predK,DF=10, eb_extent=(-0.3,0.3), cmap=CM, makepdfs=False,
                                    SNR_min=30, phaseRMS_max=30); # Ideally use 30,30!
# plt.close('all')

In [ ]:
ku0c = hologreport.generate_results(msc_Ku0, predKc, eb_extent=(-0.3,0.3), cmap=CM, makepdfs=False,
                                    SNR_min=30, phaseRMS_max=30); # Ideally use 30,30!
# plt.close('all')

In [ ]:
ku1 = hologreport.generate_results(ms_Ku1, predK,DF=10, eb_extent=(-0.3,0.3), cmap=CM, makepdfs=False,
                                    SNR_min=30, phaseRMS_max=34); # Ideally use 30,30!
# plt.close('all')

In [ ]:
ku1c = hologreport.generate_results(msc_Ku1, predKc, eb_extent=(-0.3,0.3), cmap=CM, makepdfs=False,
                                    SNR_min=30, phaseRMS_max=30); # Ideally use 30,30
plt.close('all')

In [ ]:
# The attack angles, shown centered at the elevation where the antenna was pointed
hologreport.plot_enviro(ms_Ku0+msc_Ku0, "all", "elev,wind,temp", tzoffset=2)

In [ ]:
hologreport.plot_enviro(ms_Ku1+msc_Ku1, "all", "elev,wind,temp", tzoffset=2)

In [ ]:
_ku0 = hologreport.collate_results(ku0, ku0c)
_ku1 = hologreport.collate_results(ku1, ku1c)

In [ ]:
# Find any that are incorrectly tagged as "night"
print(hologreport.filter_results(_ku0, exclude_tags=["day"], hod_filter=lambda hod: (6.5<hod<18)).keys())
print(hologreport.filter_results(_ku1, exclude_tags=["day"], hod_filter=lambda hod: (6.5<hod<18)).keys())
print()
# Find any that are incorrectly tagged as "day"
print(hologreport.filter_results(_ku0, exclude_tags=["night"], hod_filter=lambda hod: (hod<6 or 19<hod)).keys())
print(hologreport.filter_results(_ku1, exclude_tags=["night"], hod_filter=lambda hod: (hod<6 or 19<hod)).keys())

In [ ]:
# Is there a significant difference between FI0 and FI1 Y_f?
hologreport.plot_offsets_el([_ku0["night"]], ["FI0"], hide="X"); plt.xlim(15,70)
hologreport.plot_offsets_el([_ku1["night"]], ["FI1"], hide="X"); plt.xlim(15,70)

In [ ]:
print(hologreport.filter_results(ku0, elincl_deg=[41,43]).keys())

print(hologreport.filter_results(ku1, elincl_deg=[19.3,20]).keys())

In [ ]:
_ku = hologreport.collate_results(_ku0, _ku1)

hologreport.plot_vs_hod([_ku["night"]], ["FI0+FI1"], separate_freqs=False)
hologreport.plot_vs_hod([_ku["day"]], ["FI0+FI1"], separate_freqs=False)

In [ ]:
# Explore relationships
hologreport.plot_offsets_against([_ku["night"]], ["FI0+FI1"], "temp_C", hide="XY", fit='leastsq')

hologreport.plot_offsets_against([_ku["FI0"]], ["FI0"], "temp_C", hide="XZ", fit='leastsq')
hologreport.plot_offsets_against([_ku["FI1"]], ["FI1"], "temp_C", hide="XZ", fit='leastsq')
hologreport.plot_offsets_against([_ku["night"]], ["FI0+FI1"], "temp_C", hide="XZ", fit='leastsq')

hologreport.plot_offsets_against([_ku["night"]], ["FI0+FI1"], "temp_C", hide="YZ")
hologreport.plot_offsets_against([_ku["night"]], ["FI0+FI1"], "el_deg", hide="YZ", fit='leastsq');

In [ ]:
hologreport.plot_offsets_against([_ku["night"]], ["FI0+FI1"], "temp_C", hide="XY", fit='leastsq'); plt.xlim(0,30)
hologreport.plot_offsets_against([_ku["night"]], ["FI0+FI1"], "el_deg", hide="YZ", fit='leastsq'); plt.xlim(15,70);

In [ ]:
# Plots for the ATR

In [ ]:
hologreport.plot_offsets_against([_ku["night"]+_ku["day"]], ["Final FI"], "temp_C", hide="XY", fit='leastsq'); plt.xlim(0,30)
hologreport.plot_offsets_against([_ku["night"]+_ku["day"]], ["Final FI"], "el_deg", hide="YZ", fit='leastsq'); plt.xlim(15,70);

In [ ]:
__ku = hologreport.filter_results(_ku, enviro_filter=lambda e: (e['wind_mps'][2]<=7))
hologreport.plot_offsets_el([__ku["night"]+__ku["day"]], ["Final FI, PS Conditions"], hide="X", fit="theil-sen"); plt.xlim(15,70)
hologreport.plot_errbeam_el([__ku["night"]+__ku["day"]], ["Final FI, PS Conditions"]); plt.xlim(15,70);

In [ ]:
# Remove wind above STANDARD - for Error Beam spec
__ku = hologreport.filter_results(_ku, enviro_filter=lambda e: (e['wind_mps'][2]<=7))

# Also de-embed a ~1% measurement error from the Error Beam vzlues
for r in __ku["night"]+__ku["day"]:
    for fi,f in enumerate(r.f_MHz): # Each frequency
        for ebH,ebV in zip(np.atleast_2d(r.errbeamH[fi]), np.atleast_2d(r.errbeamV[fi])):
            # eb: [max,95pct,stddev,smoothing resid]
            ebH[:2] = (ebH[:2]**2 - .01**2)**.5
            ebV[:2] = (ebV[:2]**2 - .01**2)**.5

hologreport.plot_errbeam_el([__ku["night"]+__ku["day"]], ["Final FI, PS Conditions"]); plt.xlim(15,70)
plt.plot(plt.xlim(), [2, 2], 'r-', label='R.DS.P.49', alpha=0.3)
plt.plot(plt.xlim(), [4, 4], 'r--', label='R.RC.P.4', alpha=0.3); plt.legend();

In [ ]:
_ku = _ku1 # hologreport.collate_results(_ku0, _ku1)
__ku = hologreport.filter_results(_ku, enviro_filter=lambda e: (e['wind_mps'][2]<=7))

mss = ms_Ku0+msc_Ku0+ms_Ku1+msc_Ku1
for masking in [__ku]:
    freq, ant_eff, mask, fids = hologreport.calculate_ant_eff(mss, masking, return_ids=True)
    plt.figure(figsize=(14,6))
    plt.title("IEEE ANTENNA APERTURE ILLUMINATION EFFICIENCY 'As-Is' from measured patterns")
    plt.plot(freq[~mask[:,0]], ant_eff[:,0][~mask[:,0]], '_', label='H')
    plt.plot(freq[~mask[:,1]], ant_eff[:,1][~mask[:,1]], '|', label='V')
    plt.plot([8000,15000], [78,70], 'r-', alpha=0.5, label='R.D.P.1')
    plt.plot([8000,15000], [.92*78,.92*70], 'r--', alpha=0.5, label='R.D.P.9')
    plt.ylabel("$\eta_{ap}$ [%]"); plt.xlabel("f [MHz]"); plt.legend()
    plt.grid(True)

In [ ]:
# Efficiencies & associated datasets
np.c_[ant_eff[:,0],ant_eff[:,1],fids][~mask[:,0]]

# Band2

## Predicted Data

In [20]:
!ls ../models/beam-patterns/ska/B2*

../models/beam-patterns/ska/B2_45_1000.mat
../models/beam-patterns/ska/B2_45_1060.mat
../models/beam-patterns/ska/B2_45_1100.mat
../models/beam-patterns/ska/B2_45_1160.mat
../models/beam-patterns/ska/B2_45_1220.mat
../models/beam-patterns/ska/B2_45_1252.mat
../models/beam-patterns/ska/B2_45_1310.mat
../models/beam-patterns/ska/B2_45_1360.mat
../models/beam-patterns/ska/B2_45_1410.mat
../models/beam-patterns/ska/B2_45_1460.mat
../models/beam-patterns/ska/B2_45_1510.mat
../models/beam-patterns/ska/B2_45_1610.mat
../models/beam-patterns/ska/B2_45_1660.mat
../models/beam-patterns/ska/B2_45_1710.mat
../models/beam-patterns/ska/B2_45_1760.mat
../models/beam-patterns/ska/B2_45_965.mat


In [ ]:
# Predicted for L-band (max extent 120deg!)
predL = hologreport.ResultSet("predicted", f_MHz=[965,1000,1060,1310,1360,1410,1460,1660,1710], beacon_pol=[], clipextent=24)

for f_MHz,pol in zip(predL.f_MHz,predL.beacon_pol):
    b, aH, aV = hologreport.load_predicted(f_MHz, pol, pDISHPARAMS, band="B2", el_deg=45, clipextent=predL.clipextent, gridsize=512)
    predL.beams.append(b); predL.apmapsH.append(aH); predL.apmapsV.append(aV); predL.beacon_pol.append(None)

## Measured Data

In [ ]:
# Band2
rr = katselib.ls_archive(f"InstructionSet:\"scan-ant*{ant}*\" AND StartTime:[2025-01-20T0:0:0Z TO *] AND InstructionSet:\"scan-extent 24\"~2 AND InstructionSet:*holography*",
                    min_duration=1800, fields=["CaptureBlockId","StartTime","CenterFrequency",'Description','InstructionSet'], field_len=240)

In [ ]:
# FI @ -103.488deg
ms_Lband0 = [ # All of these done with 1k@1sec dump period, except where noted differently
    hologreport.ResultSet(1739268660,f_MHz=[1000,1060,1310,1360,1410,1460,1660,1710],beacon_pol=None,clipextent=10.0,tags=["FIl0","day","PKS 1934-63","0.1deg/sec"]),
    hologreport.ResultSet(1739270659,f_MHz=[1000,1060,1310,1360,1410,1460,1660,1710],beacon_pol=None,clipextent=10.0,tags=["FIl0","day","PKS J1924-2914","0.1deg/sec"]),
]

In [ ]:
### Load the data
datasets = ms_Lband1+ms_Lband0; dMHz = 10 # ~1% fractional BW 

cached_url = lambda cbid: f"./l1_data/{cbid}/{cbid}_sdp_l0.full.rdb"
for ms in datasets:
    # ms.beams.clear(); ms.apmapsH.clear(); ms.apmapsV.clear() # Un-comment to reload all
    if (len(ms.beams) > 0): continue
    ds = util.open_dataset(ms.fid, cache_root="./l1_data") # Ensure it is cached locally
    b, aH, aV = hologreport.load_data(cached_url(ms.fid), ms.f_MHz, ant, DISHPARAMS, clipextent=ms.clipextent, loadscan_cycles=ms.cycles,
                                      findwrap="findwrap" in ms.tags, extralmoffset='auto' if "extralmoffset=auto" in ms.tags else [0,0],
                                      dMHz=dMHz, polswap=ms.polswap, gridsize=512, flag_slew=True, flags_hrs=ms.flags_hrs)
    ms.beams.extend(b); ms.apmapsH.extend(aH); ms.apmapsV.extend(aV)

## Reports

In [ ]:
l = hologreport.generate_results(ms_Lband0, predL, DF=8, beampolydegree=28, beamsmoothing="zernike", eb_extent=(-2,2),
                                    SNR_min=30, phaseRMS_max=50, cmap=CM, makepdfs=False);

In [ ]:
# All
hologreport.plot_offsets_el([l["FIl0"]], ["FIl0"], fit="theil-sen")
# Just 1.3 - 1.5GHz, to match the frequency range used for pointing model fits (compare Y_f with P4)
hologreport.plot_offsets_el([hologreport.filter_results(l, fincl_MHz=(1300,1500))["FIl0"]], ["FIl0"], fit="theil-sen")

In [ ]:
l1 = hologreport.generate_results(ms_Lband1, predL, DF=8, beampolydegree=28, beamsmoothing="zernike", eb_extent=(-2,2),
                                    SNR_min=30, phaseRMS_max=50, cmap=CM, makepdfs=False);

In [ ]:
hologreport.plot_offsets_el([l1["FIl1"]], ["FIl1"], fit="theil-sen")

# S-Band

## Predicted Data

- Nothing Available

## Measured Data

In [ ]:
# Note: All S-band > 10deg were early experiments - very low edge illumination. Most of those are poorly sampled!

# S0
rr = katselib.ls_archive(f"InstructionSet:\"scan-ant*{ant}*\" AND StartTime:[2025-01-20T0:0:0Z TO *] AND CenterFrequency:[2180000000 TO 2190000000] InstructionSet:\"scan-extent 10\" AND InstructionSet:*holography*",
                    min_duration=1800, fields=["CaptureBlockId","StartTime","CenterFrequency",'Description','InstructionSet'], field_len=240)
print()
# S4
rr += katselib.ls_archive(f"InstructionSet:\"scan-ant*{ant}*\" AND StartTime:[2025-01-20T0:0:0Z TO *] AND CenterFrequency:[3061000000 TO 3062500000] AND InstructionSet:\"scan-extent 9*\" AND InstructionSet:*holography*",
                    min_duration=1800, fields=["CaptureBlockId","StartTime","CenterFrequency",'Description','InstructionSet'], field_len=240)

In [ ]:
# With FI @ -22.14deg
ms_Sband1 = [    
    hologreport.ResultSet(1733401745,f_MHz=[3401.125],beacon_pol=["RCP"],clipextent=4.0, tags=["FIs1","day","ANGOSAT 2"]),
    hologreport.ResultSet(1733444104,f_MHz=[3401.125],beacon_pol=["RCP"],clipextent=6.0, tags=["FIs1","night","ANGOSAT 2"]),
]

# With FI @ -18.55deg
ms_Sband3 = [
    hologreport.ResultSet(1738249674,f_MHz=[3409.5],beacon_pol=["RCP"],clipextent=18.0,tags=["FIs3","night","YAMAL 402","0.1deg/sec"],
                        flags_hrs=[(4/3.6,4.2/3.6), (7/3.6,7.2/3.6)]), # Aircraft RFI caused compression on scans 10 & 17
]
ms_Sband3c = [
    hologreport.ResultSet(1740006606,f_MHz=[1760,2000,2200,2300,2600],beacon_pol=None,clipextent=10,tags=["FIs3","night","Hydra A","S0","0.15deg/sec"]),
    hologreport.ResultSet(1740010564,f_MHz=[1760,2000,2200,2300,2600],beacon_pol=None,clipextent=10,tags=["FIs3","night","3C 279","S0","0.15deg/sec"]),
]

In [16]:
# S-band sub-band frequencies
for fc in [2187,2624,3062]:
    print(fc+ np.linspace(-875/2.,875/2.,3))

[1749.5 2187.  2624.5]
[2186.5 2624.  3061.5]
[2624.5 3062.  3499.5]


In [ ]:
### Load the data

# Beacon signals
# datasets = ms_Sband2+ms_Sband2b+ms_Sband3; dMHz = 0.4 # Most of the S-band beacons straddle two channels (modulation?)
# Continuum
# datasets = ms_Sband2c+ms_Sband3c; dMHz = 30 # ~1% fractional BW in S4 and at most 1.7% in S0

cached_url = lambda cbid: f"./l1_data/{cbid}/{cbid}_sdp_l0.full.rdb"
for ms in datasets:
    # ms.beams.clear(); ms.apmapsH.clear(); ms.apmapsV.clear() # Un-comment to reload all
    if (len(ms.beams) > 0): continue
    ds = util.open_dataset(ms.fid, cache_root="./l1_data") # Ensure it is cached locally
    b, aH, aV = hologreport.load_data(cached_url(ms.fid), ms.f_MHz, ant, DISHPARAMS, clipextent=ms.clipextent, loadscan_cycles=ms.cycles,
                                      findwrap="findwrap" in ms.tags, extralmoffset='auto' if "extralmoffset=auto" in ms.tags else [0,0],
                                      dMHz=dMHz, gridsize=512, flag_slew=True, flags_hrs=ms.flags_hrs)
    ms.beams.extend(b); ms.apmapsH.extend(aH); ms.apmapsV.extend(aV)

## Reports


In [ ]:
# ms_Sband2 seems un-usable; ms_Sband2 is low res & low SNR, with X & Y values fluctuating by up to 8mm - perhaps levels?
#hologreport.generate_results(ms_Sband2, pred, beampolydegree=28, beamsmoothing="zernike", SNR_min=30, phaseRMS_max=30, cmap=CM, makepdfs=False);

# Continuum (un-polarised) patterns have little/no path length asymmetry between H & V results, while GEOS (without feed corrections) are
# asymmetric. Until there are feed pattern corrections, rely on continuum for Y_f (Z_f is affected by feed pahase curvature, and X_f may be
#  affected by OffsetGregorian up-down asymmetry?
s = hologreport.generate_results(ms_Sband2b+ms_Sband2c, predS, beampolydegree=28, beamsmoothing="zernike", eb_extent=(-1.2,1.2),
                                 SNR_min=30, phaseRMS_max=30, cmap=CM, makepdfs=False);

In [ ]:
s = hologreport.generate_results(ms_Sband3c, predS, beampolydegree=28, beamsmoothing="zernike", eb_extent=(-1.2,1.2),
                                 SNR_min=20, phaseRMS_max=30, cmap=CM, makepdfs=False);

In [ ]:
hologreport.plot_offsets_el([s["FIs3"]], ["FIs3"], fit="theil-sen")

In [ ]:
s = hologreport.generate_results(ms_Sband3, predS, beampolydegree=28, beamsmoothing="zernike", eb_extent=(-1.2,1.2),
                                 SNR_min=20, phaseRMS_max=30, cmap=CM, makepdfs=False);

In [ ]:
# hologreport.meta_report(s, tags=["FIs3"])

hologreport.plot_offsets_el([s["FIs3"]], ["FIs3"], fit="theil-sen")

In [ ]:
# Un-mask the "low SNR" feed offsets; and summarise results
for k,rr in s.items():
    for r in rr:
        r.feedoffsetsH.mask = False
        r.feedoffsetsV.mask = False
hologreport.plot_offsets_el([s["FIs3"]], ["FIs3"], fit="theil-sen")